# Setup

In [ ]:
import pickle
from pathlib import Path
import pandas as pd

import keypoint_moseq as kpms

slp_project_dir = Path("/mnt/s/Projects/KY_Moseq/B1R") # customize this path to your SLP project directory
# dlc_project_dir = Path.cwd().parent / "dlc-pose-estimation" / "ElevatedMazeFood-Atanu-2026-04-04"
project_dir = Path.cwd().parent / "results" / "B1R" # customize this path to your KPMS project directory
config = kpms.load_config(str(project_dir))
model_name = project_dir / "multi_fit_20260708-13"
cleaned_snapshot = project_dir / "cleaned_keypoints.pkl"

with open(cleaned_snapshot, "rb") as f:
    snap = pickle.load(f)

coordinates = snap["coordinates"]
confidences = snap["confidences"]
bodyparts = snap["bodyparts"]

In [ ]:
FPS = 15.0
MIN_FREQUENCY = 0.01

# Assign Groups

In [ ]:
kpms.interactive_group_setting(str(project_dir), str(model_name))

# Generate dataframes

In [ ]:
moseq_df = kpms.compute_moseq_df(str(project_dir), str(model_name), fps=FPS, smooth_heading=True)
moseq_df

In [ ]:
stats_df = kpms.compute_stats_df(
    project_dir,
    model_name,
    moseq_df,
    min_frequency=MIN_FREQUENCY,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
    fps=FPS,
)  # frame rate of the video from which keypoints were inferred

stats_df

# Label syllables

In [ ]:
kpms.label_syllables(str(project_dir), str(model_name), moseq_df)

# Compare between groups

In [ ]:
kpms.plot_syll_stats_with_sem(
    stats_df,
    str(project_dir),
    str(model_name),
    plot_sig=True,  # whether to mark statistical significance with a star
    thresh=0.05,  # significance threshold
    stat="frequency",  # statistic to be plotted (e.g. 'duration' or 'velocity_px_s_mean')
    order="stat",  # order syllables by overall frequency ("stat") or degree of difference ("diff")
    ctrl_group="a",  # name of the control group for statistical testing
    exp_group="b",  # name of the experimental group for statistical testing
    figsize=(8, 4),  # figure size
    groups=stats_df["group"].unique(),  # groups to be plotted
);

## Transition matrices

In [ ]:
normalize = "bigram"  # normalization method ("bigram", "rows" or "columns")

trans_mats, usages, groups, syll_include = kpms.generate_transition_matrices(
    str(project_dir),
    str(model_name),
    normalize=normalize,
    min_frequency=MIN_FREQUENCY,  # minimum syllable frequency to include
)

kpms.visualize_transition_bigram(
    str(project_dir),
    str(model_name),
    groups,
    trans_mats,
    syll_include,
    normalize=normalize,
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)

## Syllable Transition Graph

In [ ]:
kpms.plot_transition_graph_group(
    project_dir,
    model_name,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=False,  # label syllables by index (False) or index and name (True)
)

In [ ]:
# Generate a difference-graph for each pair of groups.

kpms.plot_transition_graph_difference(
    str(project_dir), str(model_name), groups, trans_mats, usages, syll_include, layout="circular"
) 